# Phân vùng đường car6: U-Net 3 lớp, so sánh S1–S4
Notebook huấn luyện mô hình phân vùng đường cho xe car6 (vòng chung kết UCR 2026), theo quyết định train v0 ngày 01/11 (`ucr_car6_data_handover.md` mục 5, 6).

- **Đầu vào:** ảnh gốc 1280×720, cắt dải `CROP_Y` (mặc định y 320–720) rồi thu nhỏ bằng `cv2.INTER_AREA` về cỡ của biến thể; RGB, float32, chia 255 ngoài mạng.
- **Nhãn:** PNG 1280×720 do `round3_final/tools/build_seg_masks.py` tạo, giá trị 0 `background`, 1 `track_surface`, 2 `line`. Khi thu nhỏ, ô có điểm `line` giữ `line` để vạch mảnh không mất.
- **Đầu ra:** logits `[1,3,H,W]`, softmax theo kênh, lấy argmax. Vật cản là "lỗ": vùng `background` bị mặt sa bàn bao quanh.
- **Chia tập:** theo lượt, đọc `split_v0.csv` (19 lượt val). Không chia ngẫu nhiên theo ảnh.
- **Cân bằng:** trọng số lấy mẫu để mỗi lượt đóng góp tối đa `PASS_CAP` ảnh mỗi epoch.

| Mã | Kiến trúc | Đầu vào (cao×rộng) | Đo trên Nano FP16 |
|---|---|---|---|
| S1 | U-Net B-minus 8/16/32 | 96×160 | 3,8 ms |
| S2 | NanoUNet 16/32/64/128 | 96×160 | 6,8 ms |
| S3 | U-Net B-minus 8/16/32 | 128×256 | chưa đo |
| S4 | Encoder MobileNetV3-small (ImageNet) + bộ giải mã nhẹ | 128×256 | chưa đo; cần thử build TensorRT 8.2 |

Ba chế độ `MODE`:
1. `"compare"`: mỗi biến thể trong `VARIANTS` × mỗi seed trong `SEEDS`, train trên tập train, đánh giá trên val ở 5 mức sáng, ghi thêm dòng vào `Vong3/compare/road_compare.csv`. Chạy lại sau khi Colab ngắt sẽ bỏ qua lần chạy đã có trong CSV (cùng `DATA_TAG`).
2. `"final"`: FINAL_FIT, train trên toàn bộ ảnh có nhãn với `FINAL_VARIANT`, lấy bước cuối, xuất ONNX.
3. `"prelabel"`: dùng trọng số đã train để dự đoán ảnh Roboflow `car6_seg` chưa vào Dataset, đổi mặt nạ thành đa giác `track_surface`/`object` và gửi lên dạng dự đoán chờ duyệt. Lớp `glare` vẫn do người vẽ.

Chỉ số val chỉ dùng để lọc thô; quyết định cuối dựa trên FPS đo trên xe và chạy thật. `FAKE_DATA=True` tạo dữ liệu giả để kiểm quy trình trên CPU, không phản ánh chất lượng.

In [ ]:
MODE = "compare"          # "compare" | "final" | "prelabel"
FAKE_DATA = True          # True: dữ liệu giả, chạy CPU vài bước
DATA_TAG = "seg_b1"       # Đổi khi bộ nhãn đổi (vd. "seg_b2"), để không bỏ qua nhầm lần chạy cũ
VARIANTS = ["S1", "S2", "S3"]   # Thêm "S4" nếu muốn thử encoder tiền huấn luyện
SEEDS = [0, 1]
STEPS = 4000              # Số bước tối ưu mỗi lần chạy (T4: S1 ~6 phút, S3 ~10 phút)
BATCH = 32
LR = 2e-3
LINE_WEIGHT = 3.0         # Trọng số CE của lớp line
PASS_CAP = 40             # Mỗi lượt đóng góp tối đa ~40 ảnh mỗi epoch
PASTE_PROB = 0.5          # Xác suất dán vật ngẫu nhiên lên ảnh train (gán background)
FLIP_PROB = 0.5
CROP_Y = (320, 720)       # Dải hàng cắt trên ảnh 1280×720
SCAN_RATIOS = [0.60, 0.70, 0.80]   # Hàng quét theo tỉ lệ chiều cao đầu vào
BRIGHTNESS = [0.6, 0.8, 1.0, 1.2, 1.4]
HOLE_MIN_FRAC = 0.0008    # Lỗ nhỏ hơn tỉ lệ diện tích này bị bỏ qua
HOLE_HIT = 0.3            # Lỗ thật được bắt nếu >= 30% diện tích trùng lỗ dự đoán
SYNTH_PER_IMAGE = 2       # Số vật tổng hợp dán lên mỗi ảnh val khi đo bắt lỗ
EXPORT_FIRST_SEED = True  # Compare: xuất ONNX cho seed đầu của mỗi biến thể (để đo FPS)
FINAL_VARIANT = "S1"
FINAL_SEED = 0
PRELABEL_WEIGHTS = ""     # File .pth trong Vong3 (dùng cho MODE="prelabel")
PRELABEL_LIMIT = 20       # None = xử lý hết ảnh đang chờ
RF_PROJECT = "car6_seg"
RF_WORKSPACE = None
IMAGES_ZIP = "data/car6_images.zip"     # Trong Vong3: toàn bộ ảnh data_clean_car6/images
MASKS_ZIP = "data/seg_masks_v0.zip"     # Trong Vong3: PNG mặt nạ từ build_seg_masks.py
SPLIT_CSV = "data/split_v0.csv"
INSTALL_DEPS = True
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3"
WORK_DIR = "/content/work_road"         # Đĩa Colab (nhanh hơn Drive)

VARIANT_SPECS = {
    "S1": dict(arch="unet_b_minus", hw=(96, 160)),
    "S2": dict(arch="nanounet", hw=(96, 160)),
    "S3": dict(arch="unet_b_minus", hw=(128, 256)),
    "S4": dict(arch="mbv3_unet", hw=(128, 256)),
}
CLASS_NAMES = ["background", "track_surface", "line"]

## Môi trường, thư mục và hàm dùng chung
Mọi kết quả nằm trong `MyDrive/UIT_Car_Racing/Vong3/road/`; bảng so sánh chung ở `Vong3/compare/road_compare.csv`. Ảnh được giải nén ra đĩa Colab trước khi train. Khoá Roboflow lấy từ Colab Secrets `ROBOFLOW_API_KEY` hoặc nhập ẩn bằng `getpass`, không ghi ra file.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if INSTALL_DEPS:
    pkgs = ["numpy<3", "opencv-python-headless", "onnx>=1.14,<2", "onnxruntime>=1.16,<2", "onnxslim", "thop", "roboflow", "requests"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/UIT_Car_Racing/Vong3")
else:
    ROOT = Path(LOCAL_ROOT).expanduser().resolve()
    WORK_DIR = str(ROOT / "work_road")
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(ROOT / "cache" / "torch")
import re, csv, json, time, random, shutil, zipfile, hashlib, datetime, getpass
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import onnx, onnxruntime as ort
from thop import profile

DEVICE = "cuda:0" if torch.cuda.is_available() and not FAKE_DATA else "cpu"
KIND = "fake" if FAKE_DATA else "real"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "road" / KIND / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
COMPARE_CSV = ROOT / "compare" / ("road_compare.csv" if not FAKE_DATA else "road_compare_fake.csv")
COMPARE_CSV.parent.mkdir(parents=True, exist_ok=True)
WORK = Path(WORK_DIR); WORK.mkdir(parents=True, exist_ok=True)
torch.backends.cudnn.benchmark = True
if FAKE_DATA:   # chạy thử mọi biến thể, vài bước
    STEPS, BATCH, SEEDS, VARIANTS = 3, 4, SEEDS[:2], list(VARIANT_SPECS)
print("Thiết bị:", DEVICE, "| Kết quả:", OUT)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows, fields):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)

def append_csv(path, rows):
    """Ghi thêm dòng; tự mở rộng cột khi file cũ thiếu cột mới."""
    path = Path(path); old = []
    if path.exists():
        with path.open(encoding="utf-8-sig") as f: old = list(csv.DictReader(f))
    fields = list(dict.fromkeys([k for r in old + rows for k in r]))
    write_csv(path, old + rows, fields)

def done_keys(path):
    if not Path(path).exists(): return set()
    with Path(path).open(encoding="utf-8-sig") as f: return {r["run_key"] for r in csv.DictReader(f)}

def rf_project(name):
    from roboflow import Roboflow
    key = None
    if IN_COLAB:
        try:
            from google.colab import userdata; key = userdata.get("ROBOFLOW_API_KEY")
        except Exception: pass
    key = key or getpass.getpass("Khoá Roboflow (không lưu): ")
    ws = Roboflow(api_key=key).workspace(RF_WORKSPACE) if RF_WORKSPACE else Roboflow(api_key=key).workspace()
    return ws.project(name)

RF_SUFFIX = re.compile(r"_(?:jpg|jpeg|png)\.rf\.[A-Za-z0-9]+$", re.I)
NAME_RE = re.compile(r"car6_p(\d{4})_(\d{4})")
def original_stem(name):
    stem = RF_SUFFIX.sub("", Path(name).stem)
    if not NAME_RE.fullmatch(stem): raise ValueError("Tên không theo mẫu car6_pNNNN_MMMM: " + name)
    return stem

## Dữ liệu: ảnh, mặt nạ, chia train/val
Trên laptop, sau khi export COCO `car6_seg` và chạy `build_seg_masks.py --qa`, nén thư mục mặt nạ thành `seg_masks_v0.zip` (PNG ở gốc zip hoặc trong một thư mục con) và chép vào `Vong3/data/` cùng `car6_images.zip` (một lần) và `split_v0.csv`. Chỉ ảnh có mặt nạ mới được dùng. Ảnh có tên không nằm trong `split_v0.csv` làm notebook dừng.

`prep_image` và `prep_mask` là tiền xử lý duy nhất, dùng chung cho train, đánh giá, đối chiếu ONNX và prelabel; C++ trên xe phải làm đúng như `prep_image` (ghi trong `_info.json`).

In [ ]:
def prep_image(bgr, hw):
    """BGR 1280×720 -> RGB uint8 H×W: cắt CROP_Y rồi INTER_AREA."""
    if bgr.shape[:2] != (720, 1280): raise ValueError("Ảnh phải 1280×720, nhận " + str(bgr.shape))
    crop = bgr[CROP_Y[0]:CROP_Y[1]]
    return cv2.cvtColor(cv2.resize(crop, (hw[1], hw[0]), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

def prep_mask(mask, hw):
    """Mặt nạ 0/1/2 -> H×W; ô có line thì giữ line, còn lại lấy lớp chiếm nhiều hơn."""
    crop = mask[CROP_Y[0]:CROP_Y[1]]
    frac = [cv2.resize((crop == c).astype(np.float32), (hw[1], hw[0]), interpolation=cv2.INTER_AREA) for c in range(3)]
    out = (frac[1] > frac[0]).astype(np.uint8)
    out[frac[2] > 1e-3] = 2
    return out

def fake_dataset(folder, n=48):
    """Ảnh giả: thảm xám, vạch trắng, vật tối; mặt nạ cùng quy ước."""
    rng = np.random.default_rng(0); (folder / "images").mkdir(parents=True); (folder / "masks").mkdir()
    rows = []
    for i in range(n):
        p = 10 + i // 6; name = "car6_p{:04d}_{:04d}".format(p, i % 6 + 1)
        img = np.full((720, 1280, 3), (90, 80, 70), np.uint8); mask = np.zeros((720, 1280), np.uint8)
        off = int(rng.integers(-150, 150))
        mat = np.array([[200 + off, 340], [1080 + off, 340], [1280, 720], [0, 720]], np.int32)
        cv2.fillPoly(img, [mat], (120, 115, 125)); cv2.fillPoly(mask, [mat], 1)
        for x0, x1 in ((330, 60), (640, 640), (950, 1220)):
            cv2.line(img, (x0 + off, 345), (x1 + off // 2, 719), (235, 235, 235), 10)
            cv2.line(mask, (x0 + off, 345), (x1 + off // 2, 719), 2, 10)
        if i % 3 == 0:
            c = (int(rng.integers(500, 800)), int(rng.integers(450, 620)))
            cv2.ellipse(img, c, (60, 40), 0, 0, 360, (40, 60, 160), -1); cv2.ellipse(mask, c, (60, 40), 0, 0, 360, 0, -1)
        img = np.clip(img + rng.normal(0, 4, img.shape), 0, 255).astype(np.uint8)
        cv2.imwrite(str(folder / "images" / (name + ".jpg")), img); cv2.imwrite(str(folder / "masks" / (name + ".png")), mask)
        rows.append(dict(file=name + ".jpg", pass_=p, split="val" if p in (11, 15) else "train"))
    with (folder / "split.csv").open("w", newline="") as f:
        w = csv.writer(f); w.writerow(["file", "pass", "split"]); w.writerows([[r["file"], r["pass_"], r["split"]] for r in rows])

def unzip_flat(zip_path, dest, suffix, keep=None):
    """Giải nén các file đuôi `suffix` về một thư mục phẳng; bỏ qua file đã có."""
    dest.mkdir(parents=True, exist_ok=True); names = []
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            base = Path(info.filename).name
            if info.is_dir() or not base.lower().endswith(suffix) or (keep is not None and Path(base).stem not in keep): continue
            target = dest / base
            if not target.exists() or target.stat().st_size != info.file_size:
                with z.open(info) as src, target.open("wb") as dst: shutil.copyfileobj(src, dst)
            names.append(Path(base).stem)
    return names

if FAKE_DATA:
    DATA = WORK / ("fake_" + RUN_ID); fake_dataset(DATA)
    IMG_DIR, MASK_DIR, split_path = DATA / "images", DATA / "masks", DATA / "split.csv"
else:
    IMG_DIR, MASK_DIR, split_path = WORK / "images", WORK / ("masks_" + DATA_TAG), ROOT / SPLIT_CSV
    if MODE != "prelabel":
        mask_names = set(unzip_flat(ROOT / MASKS_ZIP, MASK_DIR, ".png"))
        unzip_flat(ROOT / IMAGES_ZIP, IMG_DIR, ".jpg", keep=mask_names)

with open(split_path, encoding="utf-8-sig") as f:
    SPLIT = {Path(r["file"]).stem: (int(r["pass"]), r["split"]) for r in csv.DictReader(f)}
RECORDS = []
if MODE != "prelabel":
    for mp in sorted(MASK_DIR.glob("*.png")):
        stem = original_stem(mp.name)
        if stem not in SPLIT: raise ValueError("Ảnh không có trong split_v0.csv: " + stem)
        ip = IMG_DIR / (stem + ".jpg")
        if not ip.exists(): raise ValueError("Thiếu ảnh gốc: " + ip.name)
        RECORDS.append(dict(stem=stem, image=ip, mask=mp, pass_id=SPLIT[stem][0], split=SPLIT[stem][1]))
    TRAIN = [r for r in RECORDS if r["split"] == "train"]; VAL = [r for r in RECORDS if r["split"] == "val"]
    assert TRAIN and VAL, "Cần cả ảnh train và val có mặt nạ"
    assert not {r["pass_id"] for r in TRAIN} & {r["pass_id"] for r in VAL}
    print("Ảnh có mặt nạ: train", len(TRAIN), "/ val", len(VAL), "| lượt val:", sorted({r["pass_id"] for r in VAL}))
    write_csv(OUT / "split_used.csv", [{k: r[k] for k in ("stem", "pass_id", "split")} for r in RECORDS], ["stem", "pass_id", "split"])

CACHE = {}
def load_cache(hw):
    """Tiền xử lý một lần cho mỗi cỡ đầu vào, giữ trong RAM."""
    if hw not in CACHE:
        X, Y = [], []
        for r in RECORDS:
            img = cv2.imread(str(r["image"])); m = cv2.imread(str(r["mask"]), cv2.IMREAD_UNCHANGED)
            if img is None or m is None or m.ndim != 2 or m.shape != img.shape[:2] or m.max() > 2:
                raise ValueError("Ảnh/mặt nạ sai định dạng: " + r["stem"])
            X.append(prep_image(img, hw)); Y.append(prep_mask(m, hw))
        CACHE[hw] = (np.stack(X), np.stack(Y))
    return CACHE[hw]

## Tăng cường dữ liệu và dán vật
- Ánh sáng: độ sáng ±25%, tương phản ±20%, ám màu; vùng hầm tối ám màu đèn; nhoè chuyển động; nhiễu cảm biến; vệt chói dạng nếp gấp. Lý do: phòng thi B7.08 có cửa kính, đoạn hầm, sa bàn nhăn, camera rung.
- Lật ngang ảnh và mặt nạ cùng lúc (đường không có hướng trái/phải như biển).
- **Dán vật (copy-paste):** lấy vật từ kho (lỗ thật trong mặt nạ train + hình tổng hợp, có vật màu tối giống thảm), dán lên vùng thảm, gán `background`. Mục đích: mô hình học "thảm bị che = lỗ" với vật chưa thấy.
- Kho vật cho đánh giá tổng hợp lấy từ mặt nạ val (lượt val) và hình tổng hợp seed khác, không trùng kho train.

In [ ]:
def camera_aug(x, rng):
    x = x.astype(np.float32); h, w = x.shape[:2]
    mean = x.mean(axis=(0, 1), keepdims=True)
    x = (x - mean) * rng.uniform(0.8, 1.2) + mean
    x *= rng.uniform(0.75, 1.25) * rng.uniform(0.85, 1.15, (1, 1, 3))
    if rng.random() < 0.3:   # hầm: vùng tối ám màu đèn
        x0, x1, y0 = int(rng.uniform(0, .35) * w), int(rng.uniform(.65, 1) * w), int(rng.uniform(0, .45) * h)
        x[y0:, x0:x1] *= rng.uniform(0.12, 0.4) * np.array([rng.uniform(.85, 1.2), rng.uniform(.65, 1), rng.uniform(.6, 1)])
    if rng.random() < 0.3:   # nhoè chuyển động
        k = int(rng.choice([3, 5])); kern = np.zeros((k, k), np.float32)
        if rng.random() < 0.5: kern[k // 2, :] = 1 / k
        else: kern[:, k // 2] = 1 / k
        x = cv2.filter2D(x, -1, kern)
    if rng.random() < 0.5: x += rng.normal(0, rng.uniform(1, 8), x.shape)
    if rng.random() < 0.25:  # vệt chói
        yy, xx = np.mgrid[:h, :w]
        s = xx - rng.uniform(.1, .9) * w - rng.uniform(-.4, .4) * yy
        x += np.exp(-0.5 * (s / max(1, w * rng.uniform(.015, .06))) ** 2)[..., None] * rng.uniform(40, 140)
    return np.clip(x, 0, 255).astype(np.uint8)

def find_holes(mask, min_area):
    """Lỗ = thành phần background không chạm mép trên/trái/phải (có thể chạm mép dưới)."""
    n, lab, st, _ = cv2.connectedComponentsWithStats((mask == 0).astype(np.uint8), connectivity=8)
    h, w = mask.shape; out = []
    for i in range(1, n):
        x, y, bw, bh, area = st[i]
        if area < min_area or y == 0 or x == 0 or x + bw == w: continue
        out.append((lab == i, y + bh - 1))
    return out

def hole_bank(X, Y, min_area):
    """Cắt vật (ảnh + mặt nạ vật) từ các lỗ trong mặt nạ."""
    bank = []
    for img, m in zip(X, Y):
        for comp, _ in find_holes(m, min_area):
            ys, xs = np.nonzero(comp); y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
            bank.append((img[y0:y1, x0:x1].copy(), comp[y0:y1, x0:x1].copy()))
    return bank

def synth_object(rng, hw):
    """Vật tổng hợp: elip/đa giác, màu bất kỳ hoặc xám tối giống thảm, có vân."""
    H, W = hw; oh, ow = int(rng.uniform(.08, .3) * H), int(rng.uniform(.06, .22) * W)
    oh, ow = max(oh, 4), max(ow, 4); m = np.zeros((oh, ow), np.uint8)
    if rng.random() < 0.5: cv2.ellipse(m, (ow // 2, oh // 2), (ow // 2, oh // 2), 0, 0, 360, 1, -1)
    else:
        pts = np.stack([rng.uniform(0, ow, 6), rng.uniform(0, oh, 6)], 1).astype(np.int32)
        cv2.fillPoly(m, [cv2.convexHull(pts)], 1)
    color = rng.uniform(40, 110, 3) if rng.random() < 0.35 else rng.uniform(0, 255, 3)
    img = np.clip(color + rng.normal(0, rng.uniform(3, 25), (oh, ow, 3)), 0, 255).astype(np.uint8)
    return img, m.astype(bool)

def paste(img, mask, obj, rng, hw):
    """Dán một vật lên vùng thảm; trả về mặt nạ vật (bool) đã dán hoặc None."""
    H, W = hw; o_img, o_m = obj
    s = rng.uniform(0.6, 1.4); oh, ow = max(3, int(o_m.shape[0] * s)), max(3, int(o_m.shape[1] * s))
    if oh >= H or ow >= W - 2: return None
    o_img = cv2.resize(o_img, (ow, oh), interpolation=cv2.INTER_AREA)
    o_m = cv2.resize(o_m.astype(np.uint8), (ow, oh), interpolation=cv2.INTER_NEAREST).astype(bool)
    for _ in range(10):
        y0 = int(rng.integers(int(0.15 * H), H - oh + 1)); x0 = int(rng.integers(1, W - ow))   # không chạm mép trái/phải
        under = mask[y0:y0 + oh, x0:x0 + ow][o_m]
        if (under > 0).mean() > 0.9:   # vật nằm gọn trên thảm
            img[y0:y0 + oh, x0:x0 + ow][o_m] = o_img[o_m]; mask[y0:y0 + oh, x0:x0 + ow][o_m] = 0
            full = np.zeros((H, W), bool); full[y0:y0 + oh, x0:x0 + ow] = o_m
            return full
    return None

def sample_batch(X, Y, idx, rng, hw, bank):
    xs, ys = [], []
    for i in idx:
        img, m = X[i].copy(), Y[i].copy()
        if rng.random() < PASTE_PROB:
            obj = bank[int(rng.integers(len(bank)))] if bank and rng.random() < 0.5 else synth_object(rng, hw)
            paste(img, m, obj, rng, hw)
        img = camera_aug(img, rng)
        if rng.random() < FLIP_PROB: img, m = img[:, ::-1], m[:, ::-1]
        xs.append(img); ys.append(m)
    x = torch.from_numpy(np.ascontiguousarray(np.stack(xs).transpose(0, 3, 1, 2))).float() / 255
    return x, torch.from_numpy(np.ascontiguousarray(np.stack(ys))).long()

## Kiến trúc
- `unet_b_minus` (S1, S3): U-Net 2 mức, độ rộng 8/16/32. `nanounet` (S2): 3 mức, 16/32/64/128. Giống kiến trúc đã đo FPS ngày 28/10, đầu ra đổi thành 3 kênh softmax.
- `mbv3_unet` (S4): 4 tầng đầu của MobileNetV3-small (trọng số ImageNet của torchvision, bước 2/4/8/16) và bộ giải mã nối tắt 16 kênh. Chuẩn hoá ImageNet nằm **trong** mạng; bên ngoài vẫn chỉ chia 255 như các mô hình khác. Có Hardswish và khối SE: đã kiểm xuất ONNX opset 12, còn build TensorRT 8.2 phải thử trên xe.

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, a, b):
        super().__init__(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())

class SmallUNet(nn.Module):
    def __init__(self, widths, nc=3):
        super().__init__()
        self.enc = nn.ModuleList([DoubleConv(a, b) for a, b in zip([3] + list(widths[:-1]), widths)])
        self.dec = nn.ModuleList([DoubleConv(widths[i + 1] + widths[i], widths[i]) for i in reversed(range(len(widths) - 1))])
        self.head = nn.Conv2d(widths[0], nc, 1)
    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            if i: x = F.max_pool2d(x, 2)
            x = block(x); skips.append(x)
        for block, skip in zip(self.dec, reversed(skips[:-1])):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], 1))
        return self.head(x)

class MBV3UNet(nn.Module):
    def __init__(self, nc=3, pretrained=True):
        super().__init__()
        from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
        feats = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None).features
        self.stages = nn.ModuleList([feats[0], feats[1], feats[2:4], feats[4:9]])   # bước 2, 4, 8, 16
        chans = [16, 16, 24, 48]
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.lat = nn.ModuleList([nn.Conv2d(c, 16, 1) for c in chans])
        self.fuse = nn.ModuleList([DoubleConv(32, 16) for _ in chans[:-1]])
        self.head = nn.Conv2d(16, nc, 1)
    def forward(self, x):
        size = x.shape[-2:]; x = (x - self.mean) / self.std; skips = []
        for s in self.stages: x = s(x); skips.append(x)
        y = self.lat[-1](skips[-1])
        for i in reversed(range(len(skips) - 1)):
            y = F.interpolate(y, size=skips[i].shape[-2:], mode="bilinear", align_corners=False)
            y = self.fuse[i](torch.cat([y, self.lat[i](skips[i])], 1))
        return F.interpolate(self.head(y), size=size, mode="bilinear", align_corners=False)

def build_model(arch, pretrained=True):
    if arch == "unet_b_minus": return SmallUNet((8, 16, 32))
    if arch == "nanounet": return SmallUNet((16, 32, 64, 128))
    if arch == "mbv3_unet": return MBV3UNet(pretrained=pretrained and not FAKE_DATA)
    raise ValueError(arch)

def complexity(model, hw):
    macs, _ = profile(model.cpu().eval(), inputs=(torch.zeros(1, 3, *hw),), verbose=False)
    return int(sum(p.numel() for p in model.parameters())), float(2 * macs / 1e9)

## Huấn luyện
Hàm mất mát: cross-entropy có trọng số lớp (`line` × `LINE_WEIGHT`) cộng Dice của `track_surface` và `line`. Adam, học suất giảm theo cosin. Lấy mẫu có trọng số: ảnh của lượt có `n` ảnh mang trọng số `min(1, PASS_CAP/n)`; một epoch dài bằng tổng trọng số. Giữ trọng số **bước cuối** (không chọn theo val, tránh lạc quan trên val nhỏ), cùng quy tắc với FINAL_FIT.

In [ ]:
def pass_weights(recs):
    count = {}
    for r in recs: count[r["pass_id"]] = count.get(r["pass_id"], 0) + 1
    w = np.array([min(1.0, PASS_CAP / count[r["pass_id"]]) for r in recs]); return w / w.sum(), w.sum()

def train_one(spec, seed, idx_train, X, Y, bank, log_path):
    seed_all(seed); rng = np.random.default_rng(seed); hw = spec["hw"]
    model = build_model(spec["arch"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
    cw = torch.tensor([1.0, 1.0, LINE_WEIGHT], device=DEVICE)
    probs, _ = pass_weights([RECORDS[i] for i in idx_train]); hist = []
    t0 = time.time()
    for step in range(1, STEPS + 1):
        model.train()
        pick = rng.choice(idx_train, size=BATCH, p=probs)
        x, y = sample_batch(X, Y, pick, rng, hw, bank); x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x); p = logits.softmax(1); oh = F.one_hot(y, 3).permute(0, 3, 1, 2).float()
        inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
        dice = 1 - ((2 * inter + 1) / (den + 1))[1:].mean()
        loss = F.cross_entropy(logits, y, weight=cw) + dice
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if step == 1 or step % max(1, STEPS // 10) == 0 or step == STEPS:
            hist.append(dict(step=step, loss=round(float(loss), 5), sec=round(time.time() - t0, 1)))
            print("  ", hist[-1])
    write_csv(log_path, hist, ["step", "loss", "sec"])
    return model.eval()

## Đánh giá
Mỗi mức sáng (nhân ảnh đầu vào × 0,6 … 1,4) cho một dòng CSV gồm:
- IoU từng lớp và mIoU; sai số mép trái/phải của `track_surface` và `line` trên 3 hàng quét (chia chiều rộng ảnh), số hàng mất mép/thừa.
- **Lỗ thật:** lỗ trong mặt nạ val (vật cản, người làm vật cản…). Tỉ lệ bắt theo dải hàng của điểm thấp nhất: `near` (1/3 dưới ảnh), `mid`, `far`. Số lỗ giả/100 ảnh không có lỗ thật.
- **Lỗ tổng hợp:** dán `SYNTH_PER_IMAGE` vật chưa thấy (cắt từ lỗ val + hình tổng hợp seed khác) lên ảnh val, đo tỉ lệ bắt theo dải. Dùng khi val còn ít vật thật.
Khoảng cách tạm tính theo hàng ảnh vì chưa có ảnh hiệu chuẩn A4.

In [ ]:
BANDS = ("near", "mid", "far")
def band(row, H): return "near" if row >= 2 * H / 3 else ("mid" if row >= H / 3 else "far")

@torch.no_grad()
def predict(model, x_uint8, gain=1.0):
    x = torch.from_numpy(np.ascontiguousarray(x_uint8.transpose(0, 3, 1, 2))).float() / 255
    out = []
    for i in range(0, len(x), 64):
        out.append(model((x[i:i + 64] * gain).clamp(0, 1).to(DEVICE)).argmax(1).cpu().numpy().astype(np.uint8))
    return np.concatenate(out)

def hole_stats(gt_masks, pred_masks, min_area, gt_override=None):
    hit = {b: 0 for b in BANDS}; tot = {b: 0 for b in BANDS}; false_clean = clean = false_all = 0
    for k, (g, p) in enumerate(zip(gt_masks, pred_masks)):
        H = g.shape[0]
        gh = gt_override[k] if gt_override is not None else find_holes(g, min_area)
        ph = find_holes(p, min_area)
        p_any = np.zeros_like(g, bool)
        for c, _ in ph: p_any |= c
        g_any = np.zeros_like(g, bool)
        for c, row in gh:
            g_any |= c; b = band(row, H); tot[b] += 1; hit[b] += int((c & p_any).sum() >= HOLE_HIT * c.sum())
        fp = sum(int((c & g_any).sum() < HOLE_HIT * c.sum()) for c, _ in ph); false_all += fp
        if not gh: clean += 1; false_clean += fp
    row = {"hole_n_" + b: tot[b] for b in BANDS}
    row.update({"hole_recall_" + b: (hit[b] / tot[b] if tot[b] else None) for b in BANDS})
    row.update(false_holes_per100_clean=(100 * false_clean / clean if clean else None), clean_images=clean,
               false_holes_total=false_all)
    return row

def edge_stats(gt, pred, W):
    row = {}
    for c in (1, 2):
        for ratio in SCAN_RATIOS:
            r = min(gt.shape[1] - 1, round(ratio * (gt.shape[1] - 1))); errs = []; miss = extra = 0
            for g, p in zip(gt[:, r] == c, pred[:, r] == c):
                a, b = np.flatnonzero(p), np.flatnonzero(g)
                if len(a) and len(b): errs.append((abs(int(a[0]) - int(b[0])) + abs(int(a[-1]) - int(b[-1]))) / (2 * W))
                elif len(b): miss += 1
                elif len(a): extra += 1
            key = "{}_r{:02d}".format(CLASS_NAMES[c], int(ratio * 100))
            row["edge_" + key] = float(np.mean(errs)) if errs else None
            row["miss_" + key] = miss; row["extra_" + key] = extra
    return row

def make_synth(Xv, Yv, hw, seed, bank):
    rng = np.random.default_rng(10_000 + seed); Xs, Ys, holes = [], [], []
    for img, m in zip(Xv, Yv):
        img, m = img.copy(), m.copy(); gh = list(find_holes(m, max(4, int(HOLE_MIN_FRAC * hw[0] * hw[1]))))
        for _ in range(SYNTH_PER_IMAGE):
            obj = bank[int(rng.integers(len(bank)))] if bank and rng.random() < 0.5 else synth_object(rng, hw)
            full = paste(img, m, obj, rng, hw)
            if full is not None:
                ys = np.nonzero(full)[0]; gh.append((full, int(ys.max())))
        Xs.append(img); Ys.append(m); holes.append(gh)
    return np.stack(Xs), np.stack(Ys), holes

def evaluate(model, Xv, Yv, synth, hw, min_area):
    rows = []
    for gain in BRIGHTNESS:
        pred = predict(model, Xv, gain); row = dict(brightness=gain)
        for c, n in enumerate(CLASS_NAMES):
            inter = np.logical_and(pred == c, Yv == c).sum(); union = np.logical_or(pred == c, Yv == c).sum()
            row["iou_" + n] = float(inter / union) if union else None
        row["miou"] = float(np.mean([row["iou_" + n] for n in CLASS_NAMES if row["iou_" + n] is not None]))
        row.update(edge_stats(Yv, pred, hw[1])); row.update(hole_stats(Yv, pred, min_area))
        Xs, Ys, holes = synth
        ps = predict(model, Xs, gain)
        row.update({"synth_" + k: v for k, v in hole_stats(Ys, ps, min_area, gt_override=holes).items() if "recall" in k or "hole_n" in k})
        rows.append(row)
    return rows

def save_preview(model, Xv, Yv, path, n=6):
    pal = np.array([[0, 0, 0], [0, 160, 0], [255, 255, 255]], np.uint8)
    pick = np.linspace(0, len(Xv) - 1, min(n, len(Xv))).astype(int); pred = predict(model, Xv[pick])
    tiles = [np.concatenate([Xv[i], pal[Yv[i]], pal[p]], 1) for i, p in zip(pick, pred)]
    cv2.imwrite(str(path), cv2.cvtColor(np.concatenate(tiles, 0), cv2.COLOR_RGB2BGR))

## Xuất ONNX và đối chiếu
ONNX opset 12, đầu vào cố định `[1,3,H,W]`, FP32, `onnxslim` giản lược, đầu ra logits. Đối chiếu onnxruntime với PyTorch trên ≥ 20 ảnh thật (val trước, thiếu thì thêm train): sai số xác suất softmax lớn nhất < 0,001 và argmax trùng ≥ 99,9% điểm ảnh. `_info.json` là hợp đồng cho C++ (T04b). Engine FP16 build trên xe bằng `trtexec`, không làm ở đây.

In [ ]:
def export_onnx(model, spec, name, folder, params, gflops, extra):
    import onnxslim
    H, W = spec["hw"]; path = folder / (name + ".onnx"); m = model.cpu().eval()
    torch.onnx.export(m, torch.zeros(1, 3, H, W), str(path), opset_version=12, input_names=["images"],
                      output_names=["logits"], do_constant_folding=True, dynamo=False)
    onnx.save(onnxslim.slim(onnx.load(str(path))), str(path))
    g = onnx.load(str(path)); onnx.checker.check_model(g)
    assert next(o.version for o in g.opset_import if o.domain in ("", "ai.onnx")) == 12
    assert [d.dim_value for d in g.graph.input[0].type.tensor_type.shape.dim] == [1, 3, H, W]
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    X, _ = load_cache(spec["hw"]); order = [i for i, r in enumerate(RECORDS) if r["split"] == "val"] + \
        [i for i, r in enumerate(RECORDS) if r["split"] == "train"]
    checks = []
    for i in order[:max(20, 1)]:
        x = np.ascontiguousarray(X[i:i + 1].transpose(0, 3, 1, 2), dtype=np.float32) / 255
        with torch.no_grad(): a = m(torch.from_numpy(x)).softmax(1).numpy()
        o = sess.run(None, {"images": x})[0]; b = np.exp(o - o.max(1, keepdims=True)); b /= b.sum(1, keepdims=True)
        checks.append(dict(stem=RECORDS[i]["stem"], max_prob_error=float(np.abs(a - b).max()),
                           argmax_agree=float((a.argmax(1) == b.argmax(1)).mean())))
    if len(checks) < 20 and not FAKE_DATA: raise ValueError("Cần ít nhất 20 ảnh thật để đối chiếu ONNX")
    assert max(c["max_prob_error"] for c in checks) < 1e-3 and min(c["argmax_agree"] for c in checks) >= 0.999, "ONNX lệch PyTorch"
    write_csv(folder / (name + "_onnx_check.csv"), checks, ["stem", "max_prob_error", "argmax_agree"])
    info = {"task": "segmentation", "architecture": spec["arch"], "class_names": CLASS_NAMES, "num_classes": 3,
            "mask_values": {n: i for i, n in enumerate(CLASS_NAMES)},
            "source_image_hw": [720, 1280], "crop_y": list(CROP_Y), "crop_x": [0, 1280],
            "resize": "cv2.resize INTER_AREA (crop -> input)", "input_name": "images", "input_shape": [1, 3, H, W],
            "input_hw": [H, W], "input_dtype": "float32", "color_order": "RGB", "layout": "NCHW",
            "normalization": {"scale": 1 / 255, "mean": [0, 0, 0], "std": [1, 1, 1]},
            "imagenet_normalization_in_model": spec["arch"] == "mbv3_unet", "preprocessing_in_model": False,
            "output_name": "logits", "output_shape": [1, 3, H, W], "output_type": "logits",
            "activation": "softmax_over_channels_argmax", "opset": 12, "dynamic": False, "simplified": True,
            "parameters": params, "gflops": gflops, "flops_convention": "2 FLOPs per MAC", "latency_measured": False,
            "onnx_check_images": len(checks), "onnx_max_prob_error": max(c["max_prob_error"] for c in checks),
            "fake_data": FAKE_DATA, "torch_version": torch.__version__, "onnxruntime_version": ort.__version__}
    info.update(extra); write_json(folder / (name + "_info.json"), info)
    print("  ONNX:", path)
    return path

## Chạy so sánh hoặc FINAL_FIT
`run_key = DATA_TAG|biến thể|seed`. Lần chạy đã có trong `road_compare.csv` được bỏ qua, nên khi Colab ngắt chỉ cần chạy lại từ đầu notebook. Mỗi lần chạy lưu `.pth`, nhật ký loss, ảnh xem trước (ảnh | nhãn | dự đoán) trong `road/real/<RUN_ID>/`.

In [ ]:
SUMMARY = []
if MODE in ("compare", "final"):
    names = VARIANTS if MODE == "compare" else [FINAL_VARIANT]
    seeds = SEEDS if MODE == "compare" else [FINAL_SEED]
    skip = done_keys(COMPARE_CSV) if MODE == "compare" else set()
    for v in names:
        spec = VARIANT_SPECS[v]; hw = spec["hw"]; X, Y = load_cache(hw)
        idx_tr = np.array([i for i, r in enumerate(RECORDS) if MODE == "final" or r["split"] == "train"])
        idx_va = np.array([i for i, r in enumerate(RECORDS) if r["split"] == "val"])
        min_area = max(4, int(HOLE_MIN_FRAC * hw[0] * hw[1]))
        bank_tr = hole_bank(X[idx_tr], Y[idx_tr], min_area); bank_va = hole_bank(X[idx_va], Y[idx_va], min_area)
        params, gflops = complexity(build_model(spec["arch"], pretrained=False), hw)
        print(v, spec, "| tham số", params, "| GFLOPs", round(gflops, 4), "| vật train/val", len(bank_tr), len(bank_va))
        for k, seed in enumerate(seeds):
            key = "{}|{}|{}".format(DATA_TAG, v, seed)
            if key in skip: print("Bỏ qua (đã có):", key); continue
            name = "road_{}_s{}{}".format(v, seed, "_final" if MODE == "final" else "")
            t0 = time.time(); model = train_one(spec, seed, idx_tr, X, Y, bank_tr, OUT / (name + "_loss.csv"))
            minutes = (time.time() - t0) / 60
            torch.save({"state_dict": model.state_dict(), "variant": v, **spec, "class_names": CLASS_NAMES,
                        "crop_y": CROP_Y, "seed": seed, "data_tag": DATA_TAG, "final_fit": MODE == "final"}, OUT / (name + ".pth"))
            base = dict(run_key=key, run_id=RUN_ID, data_tag=DATA_TAG, variant=v, arch=spec["arch"], input_h=hw[0],
                        input_w=hw[1], seed=seed, steps=STEPS, batch=BATCH, lr=LR, line_weight=LINE_WEIGHT, pass_cap=PASS_CAP,
                        paste_prob=PASTE_PROB, params=params, gflops=round(gflops, 5), train_min=round(minutes, 1),
                        n_train=len(idx_tr), n_val=len(idx_va), weights=str((OUT / (name + ".pth")).relative_to(ROOT)))
            if MODE == "compare":
                synth = make_synth(X[idx_va], Y[idx_va], hw, seed, bank_va)
                rows = [dict(base, **r) for r in evaluate(model, X[idx_va], Y[idx_va], synth, hw, min_area)]
                save_preview(model, X[idx_va], Y[idx_va], OUT / (name + "_preview.jpg"))
                append_csv(COMPARE_CSV, rows); SUMMARY += rows
                r1 = next(r for r in rows if r["brightness"] == 1.0)
                print("  mIoU", round(r1["miou"], 4), "| IoU line", r1["iou_line"], "| bắt lỗ thật", r1["hole_recall_near"],
                      r1["hole_recall_mid"], r1["hole_recall_far"], "| lỗ giả/100", r1["false_holes_per100_clean"])
            if MODE == "final" or (EXPORT_FIRST_SEED and k == 0):
                export_onnx(model, spec, name, OUT, params, gflops, dict(variant=v, seed=seed, data_tag=DATA_TAG,
                            final_fit=MODE == "final", trained_on="train+val" if MODE == "final" else "train"))
                model.to(DEVICE)
    print("Xong. Bảng so sánh:", COMPARE_CSV if MODE == "compare" else "(FINAL_FIT không đánh giá)")

## Tóm tắt so sánh
Trung bình ± độ lệch giữa các seed ở độ sáng 1,0 và trung bình 5 mức sáng. Chênh lệch giữa hai biến thể nhỏ hơn dao động giữa seed thì coi như không khác nhau.

In [ ]:
if MODE == "compare" and COMPARE_CSV.exists():
    import pandas as pd
    try: from IPython.display import display
    except ImportError: display = print
    df = pd.read_csv(COMPARE_CSV, encoding="utf-8-sig"); df = df[df["data_tag"] == DATA_TAG]
    cols = ["miou", "iou_track_surface", "iou_line", "edge_line_r70", "hole_recall_near", "hole_recall_mid",
            "hole_recall_far", "false_holes_per100_clean", "synth_hole_recall_near", "synth_hole_recall_mid", "synth_hole_recall_far"]
    cols = [c for c in cols if c in df]
    for title, part in (("Độ sáng 1,0", df[df["brightness"] == 1.0]), ("Trung bình 5 mức sáng", df)):
        g = part.groupby(["variant", "seed"])[cols].mean().groupby("variant").agg(["mean", "std"]).round(4)
        print(title); display(g)
    print(df.groupby("variant")[["params", "gflops", "train_min"]].first())

## Gán nhãn sơ bộ cho `car6_seg` (đợt 3)
Đặt `MODE="prelabel"`, `FAKE_DATA=False`, `PRELABEL_WEIGHTS` trỏ tới `.pth` trong Vong3. Notebook tìm ảnh `car6_seg` chưa vào Dataset, dự đoán, phóng mặt nạ về 1280×720 (phần trên `CROP_Y` coi là `background`), rồi:
- `track_surface` = đường bao ngoài của vùng `track_surface` ∪ `line` (giản lược `approxPolyDP`, sai số `SIMPLIFY_PX`);
- `object` = lỗ bên trong mặt sa bàn (cùng định nghĩa lỗ ở phần đánh giá).
Gửi bằng `save_annotation(is_prediction=True, annotation_overwrite=False)` dạng COCO một ảnh; không tự đưa vào Dataset, người duyệt và vẽ thêm `glare`. Nhật ký mã ảnh đã gửi: `road/prelabel_done.json`. Chưa thử với project Roboflow thật: chạy `PRELABEL_LIMIT=3` trước rồi mở Roboflow kiểm đa giác.

In [ ]:
SIMPLIFY_PX = 2.0
def mask_to_polygons(full_mask, min_area_px=400):
    """Mặt nạ 1280×720 -> danh sách (lớp, đa giác phẳng [x1,y1,x2,y2,...])."""
    out = []
    surf = (full_mask > 0).astype(np.uint8)
    for c in cv2.findContours(surf, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        if cv2.contourArea(c) >= min_area_px:
            out.append(("track_surface", cv2.approxPolyDP(c, SIMPLIFY_PX, True).reshape(-1).tolist()))
    for comp, _ in find_holes(full_mask, min_area_px):
        for c in cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
            out.append(("object", cv2.approxPolyDP(c, SIMPLIFY_PX, True).reshape(-1).tolist()))
    return [(k, p) for k, p in out if len(p) >= 6]

def predict_full(model, spec, bgr):
    pred = predict(model, prep_image(bgr, spec["hw"])[None])[0]
    full = np.zeros((720, 1280), np.uint8)
    full[CROP_Y[0]:CROP_Y[1]] = cv2.resize(pred, (1280, CROP_Y[1] - CROP_Y[0]), interpolation=cv2.INTER_NEAREST)
    return full

def coco_single(name, polys):
    cats = {"track_surface": 1, "object": 2}
    anns = []
    for i, (k, p) in enumerate(polys, 1):
        xs, ys = p[0::2], p[1::2]
        anns.append(dict(id=i, image_id=1, category_id=cats[k], segmentation=[p], iscrowd=0,
                         area=float(cv2.contourArea(np.array(p, np.float32).reshape(-1, 2))),
                         bbox=[min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]))
    return dict(images=[dict(id=1, file_name=name, width=1280, height=720)], annotations=anns,
                categories=[dict(id=v, name=k, supercategory="none") for k, v in cats.items()])

if MODE == "prelabel" or FAKE_DATA:
    import requests, inspect
    if FAKE_DATA:
        spec = VARIANT_SPECS[VARIANTS[0]]; model = build_model(spec["arch"], pretrained=False).to(DEVICE).eval()
        pending = [dict(id=r["stem"], name=r["stem"] + ".jpg", local=r["image"]) for r in RECORDS[:2]]
    else:
        wp = Path(PRELABEL_WEIGHTS).expanduser().resolve()
        if not wp.is_file() or ROOT not in wp.parents: raise ValueError("PRELABEL_WEIGHTS phải là file .pth trong Vong3")
        ck = torch.load(wp, map_location=DEVICE, weights_only=False)
        if tuple(ck["crop_y"]) != tuple(CROP_Y): raise ValueError("CROP_Y khác lúc train")
        spec = dict(arch=ck["arch"], hw=tuple(ck["hw"])); model = build_model(spec["arch"], pretrained=False).to(DEVICE)
        model.load_state_dict(ck["state_dict"]); model.eval()
        project = rf_project(RF_PROJECT); pending = []; offset = 0
        while True:
            page = project.search(in_dataset=False, offset=offset, limit=250, fields=["id", "name"])
            if not isinstance(page, list): raise RuntimeError("Định dạng search Roboflow thay đổi")
            pending += page; offset += 250
            if len(page) < 250: break
    journal = ROOT / "road" / "prelabel_done.json"
    done = json.loads(journal.read_text(encoding="utf-8")) if journal.exists() and not FAKE_DATA else []
    prefix = "{}/{}:".format(RF_WORKSPACE or "default", RF_PROJECT)
    pending = [p for p in pending if prefix + str(p["id"]) not in set(done)]
    if PRELABEL_LIMIT is not None: pending = pending[:PRELABEL_LIMIT]
    PRE = OUT / "prelabel"; PRE.mkdir(exist_ok=True); logs = []
    for item in pending:
        if FAKE_DATA: bgr = cv2.imread(str(item["local"]))
        else:
            data = project.image(item["id"]); urls = data.get("urls") or {}
            url = urls.get("original") or data.get("image_url")
            if not url: raise RuntimeError("Roboflow không trả URL ảnh")
            resp = requests.get(url, timeout=(10, 30)); resp.raise_for_status()
            bgr = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None or bgr.shape[:2] != (720, 1280): raise RuntimeError("Ảnh không đọc được hoặc không phải 1280×720: " + str(item["name"]))
        polys = mask_to_polygons(predict_full(model, spec, bgr))
        path = PRE / (hashlib.sha256(str(item["id"]).encode()).hexdigest()[:20] + ".json")
        write_json(path, coco_single(item["name"], polys)); status = "khong_co_da_giac"
        if polys:
            if FAKE_DATA: status = "gia_lap"
            else:
                kw = dict(annotation_path=str(path), image_id=item["id"], is_prediction=True, annotation_overwrite=False)
                if "add_to_dataset" in inspect.signature(project.save_annotation).parameters: kw["add_to_dataset"] = False
                project.save_annotation(**kw)
                status = "da_gui_can_duyet"; done.append(prefix + str(item["id"])); write_json(journal, sorted(set(done)))
        logs.append(dict(image_id=item["id"], name=item["name"], polygons=len(polys), status=status))
        write_csv(PRE / "prelabel_log.csv", logs, ["image_id", "name", "polygons", "status"])
    print("Ảnh xử lý:", len(logs), "| Mở Roboflow để duyệt, vẽ glare nếu cần.")